In [ ]:
import re
from pprint import pprint
from pathlib import Path


In [ ]:
DAY = r"\d{1,2}"
MONTH_NUM = r"\d{1,2}"
MONTH_NAME = r"(Januar|Februar|März|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember)"
YEAR = r"\d{2,4}"

DATE = rf"""
(?:
    \b{DAY}[./]{MONTH_NUM}[./]{YEAR}\b        |  # 21.09.2007 / 12/12/66
    \b{YEAR}-\d{{2}}-\d{{2}}\b                |  # 2027-08-02
    \b{DAY}\.\s*{MONTH_NAME}\s*{YEAR}\b       |  # 27. März 2025
    \*\s*{DAY}[./]{MONTH_NUM}[./]{YEAR}       |  # * 3.2.1961
    \b{DAY}\.\s*{MONTH_NUM}\.\b               # 2.3.
)
"""

DATE_RANGE = rf"""
\b
{DAY}[./]{MONTH_NUM}[./]{YEAR}
\s*(\-|–|bis|zum)\s*
{DAY}[./]{MONTH_NUM}[./]{YEAR}
\b
"""

ID = r"\b(?:Patienten-ID|PIZ|Fallnummer|Vorgangs-Nr\.?)\s*:\s*\d+\b"

EMAIL = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
PHONE = r"\+?\d[\d\s\-()/]{6,}"


In [ ]:
def regex_mask_text(text):
    # Reihenfolge ist entscheidend
    text = re.sub(DATE_RANGE, "[DATE_RANGE]", text, flags=re.IGNORECASE | re.VERBOSE)
    text = re.sub(DATE, "[DATE]", text, flags=re.IGNORECASE | re.VERBOSE)
    text = re.sub(ID, "[ID]", text, flags=re.IGNORECASE)
    text = re.sub(EMAIL, "[EMAIL]", text)
    text = re.sub(PHONE, "[PHONE]", text)
    return text


In [ ]:
def create_regex_dictionary(docs: dict) -> dict:
    new_docs = {}

    for doc_idx, doc in docs.items():
        new_docs[doc_idx] = {
            "doc_id": doc["doc_id"],
            "text_original": doc["text"],
            "text_regex": regex_mask_text(doc["text"]),
            "spans": doc["spans"] 
        }

    return new_docs


In [ ]:
import json

# 1. Load original list
with open("data_regex/processed/grascco_canonical.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

# 2. Convert list -> indexed dict
docs = {i: doc for i, doc in enumerate(docs)}

# 3. Apply regex preprocessing
docs_regex = create_regex_dictionary(docs)

# 4. Convert dict -> list (same structure as original)
docs_regex_list = [
    {
        "doc_id": doc["doc_id"],
        "text": doc["text_regex"],          # regex version becomes main text
        "text_original": doc["text_original"],
        "spans": doc["spans"]
    }
    for _, doc in sorted(docs_regex.items())
]

# 5. Save as NEW file
with open("data_regex/processed/grascco_canonical_regex.json", "w", encoding="utf-8") as f:
    json.dump(docs_regex_list, f, ensure_ascii=False, indent=2)
